# LangChain Agents with Custom Middleware

This notebook demonstrates advanced capabilities of LangChain agents utilizing custom middleware. It includes detailed examples of:
1. **Summarization Middleware** (triggered by message count and token count) for managing long context history.
2. **Human-In-The-Loop Middleware** for pausing execution to request approval or edit tool call arguments before execution.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_BASE_URL"] = os.getenv('OPENAI_BASE_URL')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


## 1. Summarization Middleware (Message-based Trigger)

In this section, we set up an agent with a `SummarizationMiddleware` triggered when the number of messages in the chat history exceeds 10. It collapses older messages into a concise summary while preserving the most recent 4 messages.

# Summarization Middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model ="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware= [SummarizationMiddleware(
        model = "gpt-4o-mini",
        trigger=("messages",10),
        keep=("messages",4))]
)


### Running the Agent with Thread ID and Message History

We define a test configuration with a persistent thread ID and simulate a multi-turn conversation about Paris Saint-Germain (PSG) to trigger the message-based summarization threshold.

In [ ]:
##Run with thread-id
config = {"configurable": {"thread_id":"test1"}}

In [ ]:
questions = [
    "Which city is the capital of France?",
    "Has PSG ever won the Champions League?",
    "Did Cristiano Ronaldo play for PSG?",
    "Who is the current manager of PSG?",
    "Who is the most successful manager at PSG?",
    "Who has scored the most goals for PSG?",
    "Who has provided the most assists for PSG?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Which city is the capital of France?', additional_kwargs={}, response_metadata={}, id='929bc874-2eaa-4294-94aa-cbd6f7366a98'), AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 15, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.00073828806183, 'cost_details': {'completion_cost_inr': 0.00048074571468, 'prompt_cost_inr': 0.00025754234715}, 'latency_ms': 720}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_19227531c6', 'id': 'chatcmpl-EVIwpsH80cQyh6vsWrq0P0ExqL5V1', 'service_tier': 'default', 'finish_reason':

In [ ]:
response['messages']

[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to obtain information about Paris Saint-Germain (PSG) and related football queries.\n\n## SUMMARY\nThe user asked about the capital of France, which is Paris. They inquired if PSG has ever won the UEFA Champions League, to which it was confirmed that, as of October 2023, PSG has not won the title, with their closest attempt being a final loss in the 2019-2020 season against Bayern Munich. The user also asked if Cristiano Ronaldo has played for PSG; the response noted that he has not, detailing his career at other clubs instead, including his transfer to Al Nassr in December 2022.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nContinue answering user queries related to PSG, including the current manager inquiry.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='a6729b0d-42e0-4396-8202-311cde57d5f7'),
 AIMessage(content='As of October 2023, the manag

## 2. Summarization Middleware (Token-based Trigger)

Instead of checking the absolute message count, we can trigger context summarization when the total token count of the conversation exceeds a specific limit. Here, we simulate hotel searches to quickly accumulate tokens.

### Testing Token-Based Triggers

We run a series of hotel search requests across multiple global destinations. As the returned descriptions accumulate, the middleware automatically triggers summarization once the thread passes 550 tokens.

# TokenSize

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")



Paris: ~143 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='06eea813-6b4d-4f86-a0f2-b7b000d40eb0'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.00194015234853, 'cost_details': {'completion_cost_inr': 0.0010301693886, 'prompt_cost_inr': 0.0009099829599300001}, 'latency_ms': 953}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_adf63624f7', 'id': 'chatcmpl-EVJ8uCOInrqjif1liyOv1uEYQqVBc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 

# HumanInTheLoop Middleware

## 3. Human-In-The-Loop Middleware (Action Approval & Editing)

This section showcases how to intercept sensitive actions (such as sending emails) before they run. The middleware halts agent execution, raising an interrupt that requires an explicit resume command from the user.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by it's ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recepient: str, subject: str, body: str) -> str:
    """ Mock function to send an email."""
    return f"email sent to {recepient} with subject {subject}"



### Defining the Agent with Interrupt Middleware

We construct an agent using `HumanInTheLoopMiddleware` set to interrupt specifically on the `send_email_tool` but bypass interrupts on safe tools like `read_email_tool`.

In [ ]:
agent =  create_agent(
    model="gpt-4o",
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
    interrupt_on = {
        "send_email_tool":{
            "allowed_decisions":["approve", "edit", "reject"]
        },
        "read_email_tool":False,
    }
    )]
)


In [ ]:
from langchain_core.messages import HumanMessage
config = {"configurable":{"thread_id": "test-approve"}}
#Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="send email to Cbale@test.com with subject 'Hello' and body 'How are you buddy?'" )]},
    config=config
)

### Scenario A: Approving an Action Request

We send a command to draft an email. We verify that the agent execution pauses and check the metadata of the returned interrupt object.

In [ ]:
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'recepient': 'Cbale@test.com', 'subject': 'Hello', 'body': 'How are you buddy?'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'recepient': 'Cbale@test.com', 'subject': 'Hello', 'body': 'How are you buddy?'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='9e328226a2b6ba2593d79e59216ef1d9')]

### Resuming with Approval

We feed an approval signal back to the agent thread using LangGraph's `Command` class to resume execution and successfully trigger the email function.

In [ ]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("paused approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"} ##use "reject" to reject
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

paused approving
Result: The email has been sent to Cbale@test.com with the subject "Hello" and the body "How are you buddy?".


### Editing

### Scenario B: Editing Action Arguments on the Fly

In this scenario, a tool call request contains incorrect details (e.g., a typo in the recipient's email address). We intercept the call and edit the arguments dynamically prior to resumption.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [ ]:


config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)



In [ ]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='265cd0da-b0b6-40d0-900a-40328ea72e10'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.05583042718110001, 'cost_details': {'completion_cost_inr': 0.028630988298000003, 'prompt_cost_inr': 0.027199438883100006}, 'latency_ms': 975}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_02e43187ba', 'id': 'chatcmpl-EVZByckpcxylVlYnTDw28H8zufM5w', 'service_tier': 'default', 'finish_reason': 

In [ ]:
if '__interrupt__' in result:
    print("Paused for editing")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type" : "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args":{
                            "recipient" : "Cbale@gmail.com",
                            "subject" : "Inquiry for availabilty",
                            "body" : "I am making a movie about Singularity and Quantum Computers, I want to pitch my project to you. Are you avaiable?"
                        }
                    }}
                ]
            }
        ),
        config=config
    )

    print(f"✏️ Result: {result['messages'][-1].content}")



Paused for editing
✏️ Result: The email has been successfully sent to the recipient with the subject "Inquiry for availabilty." If you have any more requests or need further assistance, feel free to let me know!


In [ ]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='265cd0da-b0b6-40d0-900a-40328ea72e10'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'cost': 0.05583042718110001, 'cost_details': {'completion_cost_inr': 0.028630988298000003, 'prompt_cost_inr': 0.027199438883100006}, 'latency_ms': 975}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_02e43187ba', 'id': 'chatcmpl-EVZByckpcxylVlYnTDw28H8zufM5w', 'service_tier': 'default', 'finish_reason': 